In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Install Cellpose-SAM

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
import numpy as np
import os, sys
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from natsort import natsorted

### Faltou:

sudo apt install nvidia-cuda-toolkit


In [ ]:
!lspci | grep -i nvidia

In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi

In [ ]:
# reinstall cellpose to see currect nvidia version - is necessary?
# !pip3 install cellpose --force
# Successfully uninstalled cellpose-4.0.7

In [ ]:
import torch
torch.__version__, np.__version__

In [ ]:
torch.cuda.is_available()

In [ ]:
torch.cuda.device_count()

In [ ]:
torch.cuda.get_device_name(0), torch.version.cuda

In [ ]:
# !pip3 install cellpose --force

In [ ]:
from cellpose import models, core, io, plot

#### run this to get printing of progress

In [ ]:
io.logger_setup()

#### Check if colab notebook instance has GPU access

In [ ]:
if core.use_gpu()==False:
    print(ImportError("No GPU access, change your runtime"))
else:
    print("GPU Ok")

### Testing Matrix multiplication

In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

B, H, N, D = 2, 4, 128, 64
q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float32)
k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float32)
scale = (D ** -0.5)

try:
    out = (q * scale) @ k.transpose(-2, -1)
    print("FP32 batched matmul OK:", out.shape)
except Exception as e:
    print("FP32 batched matmul FAILED:", repr(e))

### CellposeModel

  - folder: /home/flalix/.cellpose/models/

In [ ]:
# torch.set_default_dtype(torch.float32)  # force float32 math
old_code = False

if old_code:
    del(torch)
    
    import torch
    
    class _NoAutocast:
        def __init__(self, enabled=False): pass
        def __enter__(self): return None
        def __exit__(self, exc_type, exc, tb): return False
    
    torch.cuda.amp.autocast = _NoAutocast   # global monkey-patch: autocast becomes a no-op
    
    torch.set_default_dtype(torch.float32)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

model = models.CellposeModel(gpu=True, model_type='cyto')

In [ ]:
root0 = "../../colaboracoes/deOcesano"
os.listdir(root0)

In [ ]:
root_samples = os.path.join(root0, 'samples')
root_plates = os.path.join(root_samples, 'Plate1848')
plates = os.listdir(root_plates)
print(root_plates)
plates

In [ ]:
root_1perc = os.path.join(root_plates, '1% SFB')
os.path.exists(root_1perc), root_1perc

In [ ]:
files = os.listdir(root_1perc)
for fname in files[:5]:
    print(fname)

In [ ]:
i=0
filefig = os.path.join(root_1perc, files[i])
img = io.imread(filefig)

fig = plt.figure(figsize=(8,8))
plt.imshow(img);

In [ ]:
type(img)

In [ ]:
img.shape

In [ ]:
type(img[:,0,0])

In [ ]:
[type(x) for x in img[:5,0,0]]

In [ ]:
[type(x) for x in img[0,:5,0]]

In [ ]:
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '2' # @param ['None', 0, 1, 2, 3, 4, 5]

In [ ]:
selected_channels = []
for i, c in enumerate([first_channel, second_channel, third_channel]):
  if c == 'None':
    continue
  if int(c) > img.shape[-1]:
    assert False, 'invalid channel index, must have index greater or equal to the number of channels'
  if c != 'None':
    selected_channels.append(int(c))

selected_channels

### Error: root cause

cublasGemmStridedBatchedEx is called when PyTorch uses half precision (float16) GEMMs.

On Maxwell GPUs (compute 5.2), these functions are not implemented in cuBLAS.

Even if you set torch.set_default_dtype(torch.float32), autocast or mixed precision inside Cellpose can override it.


### Why it fails

Many official PyTorch CUDA-12.x binaries do not include Maxwell (sm_52) in their prebuilt GPU capability list. When you run torch, it warns: “CUDA capability sm_52 is not compatible with the current PyTorch installation” (this is exactly what users see when they install a cu126 wheel). 
GitHub

NVIDIA’s CUDA toolkits have deprecated Maxwell (sm_52) support in recent versions and later removed offline/library support in the 13.x series; CUDA 12.x still works but Maxwell is deprecated, meaning upstream toolchains and binaries may stop targeting it. That’s why modern PyTorch wheels built for CUDA 12.6 often omit sm_52.


In [ ]:
hasattr(model, 'net')

In [ ]:
if old_code:
    if hasattr(model, 'net'):
        # move to cuda then convert to float
        model.net = model.net.to('cuda').float()
        # sanity check: all params should be float32
        for n, p in list(model.net.named_parameters())[:5]:
            assert p.dtype == torch.float32, f"param {n} is {p.dtype}"
        print("Model parameters converted to:", next(model.net.parameters()).dtype)

In [ ]:
%%time

img_selected_channels = np.zeros_like(img)
img_selected_channels[:, :, :len(selected_channels)] = img[:, :, selected_channels]

flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

masks, flows, styles = model.eval(
    img_selected_channels,
    batch_size=32,
    flow_threshold=flow_threshold,
    cellprob_threshold=cellprob_threshold,
    normalize={"tile_norm_blocksize": tile_norm_blocksize}
)


In [ ]:
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))

In [ ]:
fig = plt.figure(figsize=(14,8))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()

In [ ]:
if old_code:
    net = getattr(model, 'net', None)
    if net is None:
        raise RuntimeError("Could not find model.net — adjust this to your model object")
    print(net)
    
    # move to GPU and convert everything to float32
    net.to('cuda')
    for name, p in net.named_parameters(recurse=True):
        if p.dtype != torch.float32:
            p.data = p.data.float()
    for name, buf in net.named_buffers(recurse=True):
        if buf.dtype != torch.float32:
            # some buffers might be empty tensors; guard against that
            if buf.numel() > 0:
                buf.data = buf.data.float()
    
    # sanity check
    dtypes = set(p.dtype for p in net.parameters())
    if dtypes != {torch.float32}:
        print("Warning: not all params are float32:", dtypes)
    else:
        print("All model parameters are float32")
    
    print("\n-----------------\n")
    print(net)

In [ ]:
# !python run_cellpose_fp32.py